# Tutorial de execução — PQC-IoT Simulator

Neste exemplo, vamos montar uma rede IoT, executar uma simulação no Mininet-WiFi e analisar consumo de energia, tempo, tráfego e entrega.

## 1. Inicie o ambiente

Na raiz do repositório, execute:

```bash
sudo docker compose run --rm --service-ports notebook
```

Abra a URL mostrada no terminal e selecione o kernel **Python 3 - PQC IoT Simulator**.

## 2. Importe a plataforma

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pqc_iot_simulator.pqc_iot_sim import Network, Simulation

## 3. Crie a rede

Neste exemplo usamos uma grade com três linhas e três colunas. A grade 3 × 3 possui nove posições: sete nós IoT, um gateway e um servidor.

Você pode substituir a chamada por qualquer uma destas topologias:

| Topologia | Parâmetros | Exemplo |
|---|---|---|
| `grade` | `rows`: linhas; `cols`: colunas; `snake_path`: inclui um caminho sequencial adicional | `rede.set_ready_topology("grade", rows=3, cols=3, snake_path=False)` |
| `star` | `nodes`: quantidade de dispositivos IoT ligados ao gateway | `rede.set_ready_topology("star", nodes=5)` |
| `tree` | `levels`: níveis abaixo do gateway; `children`: filhos por nó | `rede.set_ready_topology("tree", levels=2, children=2)` |
| `mesh` | `nodes`: quantidade de dispositivos IoT; `mesh_degree`: vizinhos IoT por nó | `rede.set_ready_topology("mesh", nodes=5, mesh_degree=2)` |

Todos os números devem ser inteiros positivos. Na grade, `rows × cols` deve ser pelo menos 3 para acomodar um dispositivo IoT, o gateway e o servidor. Use somente uma chamada de topologia por cenário.

In [ ]:
rede = Network()
rede.set_ready_topology("grade", 3, 3)
rede.draw()

## 4. Escolha a segurança e o protocolo

O primeiro argumento de `set_crypto_mode` seleciona o conjunto criptográfico:

| Valor | Estabelecimento e autenticação |
|---|---|
| `classical` | X25519 e Ed25519 |
| `pqc` | ML-KEM-512 e ML-DSA-44 |
| `hybrid` | combinação dos algoritmos clássicos e pós-quânticos |

O parâmetro `handshake_variant` aceita:

- `preprovisioned`: o servidor já possui credenciais provisionadas; é a opção mais simples;
- `full_certificate`: inclui certificados e enquadramento adicional no handshake.

Exemplos:

```python
rede.set_crypto_mode("pqc", handshake_variant="preprovisioned")
rede.set_crypto_mode("hybrid", handshake_variant="full_certificate")
```

O protocolo pode ser `mqtt`, `coap` ou `http`:

```python
rede.set_protocol("mqtt")
rede.set_protocol("coap")
rede.set_protocol("http")
```

Use apenas uma escolha de criptografia e uma escolha de protocolo em cada cenário.

In [ ]:
rede.set_crypto_mode("classical", handshake_variant="preprovisioned")
rede.set_protocol("mqtt")

Confira as escolhas feitas para a rede.

Se quiser selecionar métricas explicitamente, use `rede.set_metrics(...)`. Os nomes aceitos são:

`latency`, `jitter`, `packet_loss`, `throughput`, `pdr`, `energy`, `crypto_time`, `message_overhead`, `link_metrics`, `rtt` e `real_latency`.

Exemplo:

```python
rede.set_metrics(["latency", "packet_loss", "energy", "crypto_time"])
```

Quando esse método não é chamado, a plataforma usa o conjunto padrão.

In [ ]:
rede.summary()

## 5. Configure a simulação

Vamos usar o perfil `esp32_bme280_wifi` no nó `iot_1`, enviando dados para `server_1`.

Em `set_devices`:

- os primeiros argumentos são os nós IoT que participarão da simulação;
- `profile` seleciona o perfil físico;
- `destination` deve indicar um servidor existente na topologia.

É possível selecionar mais de um dispositivo:

```python
simulacao.set_devices(
    "iot_1",
    "iot_2",
    profile="esp32_bme280_wifi",
    destination="server_1",
)
```

Os dispositivos precisam existir na topologia escolhida. Para consultar os perfis disponíveis, execute:

```python
simulacao.available_device_profiles()
```

In [ ]:
simulacao = Simulation(rede)
simulacao.set_devices("iot_1", profile="esp32_bme280_wifi", destination="server_1")

Defina a duração do cenário e o intervalo entre amostras:

- `set_duration(seconds)`: tempo total representado pela simulação; deve ser maior que zero;
- `set_sample_interval(seconds)`: intervalo entre duas leituras do sensor; também deve ser maior que zero.

Com duração de 60 segundos e intervalo de 60 segundos, o dispositivo executa um ciclo. Para gerar mais ciclos, aumente a duração ou diminua o intervalo. Por exemplo:

```python
simulacao.set_duration(600)
simulacao.set_sample_interval(60)  # uma leitura por minuto
```

In [ ]:
simulacao.set_duration(60)
simulacao.set_sample_interval(60)

Configure a bateria e a política de renovação da sessão criptográfica.

Parâmetros da bateria:

- `capacity_mah`: capacidade nominal em mAh; deve ser maior que zero;
- `initial_soc_percent`: carga inicial entre 0 e 100%.

Parâmetros da sessão:

- `lifetime_seconds`: validade máxima da sessão; deve ser maior que zero;
- `max_messages`: quantidade máxima de mensagens protegidas pela mesma sessão; deve ser maior que zero.

A sessão é renovada quando o limite de tempo ou de mensagens é alcançado, prevalecendo o que ocorrer primeiro.

In [ ]:
simulacao.set_battery(capacity_mah=2000, initial_soc_percent=100)
simulacao.set_session(lifetime_seconds=3600, max_messages=100)

Defina as condições configuradas do enlace:

- `bandwidth_mbps`: largura de banda em Mbit/s; deve ser maior que zero;
- `latency_ms`: atraso de ida em milissegundos; não pode ser negativo;
- `packet_loss_percent`: perda de pacotes entre 0 e 100%.

Exemplo de um enlace mais restrito:

```python
simulacao.set_link(
    bandwidth_mbps=2,
    latency_ms=40,
    packet_loss_percent=5,
)
```

Durante a execução, o Mininet-WiFi aplica esses valores ao enlace e a plataforma também registra as métricas que conseguir observar.

In [ ]:
simulacao.set_link(
    bandwidth_mbps=10,
    latency_ms=5,
    packet_loss_percent=0,
)

Selecione a calibração criptográfica e o diretório de saída.

Perfis bibliográficos do ESP32 disponíveis:

- `esp32_literatura_inferior.csv`, `esp32_literatura_central.csv` e `esp32_literatura_superior.csv`;
- `esp32_literatura_alternativa_inferior.csv`, `esp32_literatura_alternativa_central.csv` e `esp32_literatura_alternativa_superior.csv`.

Os cenários `inferior`, `central` e `superior` representam limites e ponto médio de parâmetros bibliográficos; não são medições do seu ESP32. Também é possível fornecer um CSV próprio com medições futuras.

`set_calibration(caminho, strict=True)` exige por padrão todas as operações necessárias. Use `strict=False` somente para estudos exploratórios que aceitem operações sem calibração.

`set_output(diretorio)` define onde serão gravados JSON, CSV e PCAP. Use uma pasta diferente para cada cenário para não sobrescrever resultados anteriores.

In [ ]:
simulacao.set_calibration("calibracao/esp32_literatura_central.csv")
simulacao.set_output("results/raw/tutorial")

## 6. Execute

A execução inicia o Mininet-WiFi, associa os nós, transmite as mensagens e encerra a rede ao final.

Antes de executar, você pode voltar às células anteriores e alterar qualquer parâmetro. Depois de uma alteração, execute novamente essa célula e todas as células seguintes.

In [ ]:
resultado = simulacao.run()
ciclos = pd.DataFrame(resultado["cycles"])

## 7. Confira a execução real

PCAP é um arquivo de captura de pacotes. Ele registra o tráfego observado na interface de rede e permite inspecioná-lo posteriormente em ferramentas como Wireshark ou `tcpdump`.

In [ ]:
ciclos[[
    "device_id",
    "mininet_executed",
    "pcap_capture_available",
    "delivered",
]]

Visualize as principais métricas do ciclo.

In [ ]:
ciclos[[
    "device_id",
    "crypto_mode",
    "application_protocol",
    "mininet_transport_duration_seconds",
    "total_wire_bytes",
    "energy_crypto_j",
    "energy_total_capture_coupled_j",
]]

## 8. Compare o consumo por componente

In [ ]:
componentes = {
    "energy_sensor_j": "Sensor",
    "energy_cpu_j": "CPU",
    "energy_crypto_j": "Criptografia",
    "energy_tx_j": "Transmissão",
    "energy_rx_j": "Recepção",
    "energy_sleep_j": "Sono",
}

energia = ciclos[list(componentes)].sum().rename(index=componentes)
energia.plot.bar(figsize=(9, 4), color="#1f4e79")
plt.ylabel("Energia (J)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 9. Resultados gerados

A execução salva automaticamente:

- `results/raw/tutorial/ciclos.csv` — métricas por ciclo;
- `results/raw/tutorial/resultado.json` — resumo e resultados completos;
- `results/raw/tutorial/pcap/` — capturas de pacotes.

Para criar outro cenário, altere uma ou mais opções:

- topologia: `grade`, `star`, `tree` ou `mesh`;
- criptografia: `classical`, `pqc` ou `hybrid`;
- handshake: `preprovisioned` ou `full_certificate`;
- protocolo: `mqtt`, `coap` ou `http`;
- dispositivos e perfil físico;
- duração, intervalo de amostragem, bateria e sessão;
- banda, latência e perda;
- calibração e diretório de saída.

Execute novamente a partir da primeira célula alterada. Para comparar cenários, salve cada execução em uma pasta diferente.